In [ ]:
import requests
import time
import json
from transformers import Qwen2TokenizerFast, AutoTokenizer
from tqdm import tqdm
TOKENIZER: Qwen2TokenizerFast = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-0.5B", use_fast=True)
def get_length_function(text: str) -> int:
    return TOKENIZER(text, return_tensors=None, add_special_tokens=False).input_ids.__len__()

In [ ]:
URL = "http://localhost:11434/api/generate"
PREFIX = "<|fim_prefix|>"
SUFFIX = "<|fim_suffix|>"
MIDDLE = "<|fim_middle|>"
DISPLAY_LIMIT = 1024 # <= 1024, for display result
DISPLAY_SAMPLE_STEP = 8  # For display result
save_file_name = "benchmark_1024_all_stop_4-3-16.pkl"
model_names = [
    "qwen25-coder-05b-instruct:latest",
    "qwen25-coder-05b-instruct-q4:latest",
    "qwen25-coder-15b-instruct:latest",
    "qwen25-coder-15b-instruct-q4:latest",
]

In [ ]:
def benchmark(url: str, model: str, prompt: str, session):
    first_token_ns = 0
    last_token_ns = 0
    first_byte_ns = 0
    payload = {
        "model": model,
        "prompt": prompt,
        "raw": True,
        "options": {
            "temperature": 0.01,
            "num_predict": 16,
            # "stop": [
            #     "\n",
            #     "\n\n",
            #     "<|endoftext|>",
            #     "<|fim_prefix|>",
            #     "<|fim_middle|>",
            #     "<|fim_suffix|>",
            #     "<|fim_pad|>",
            #     "<|repo_name|>",
            #     "<|file_sep|>",
            #     "<|im_start|>",
            #     "<|im_end|>",
            #     "/src/",
            #     "#- coding: utf-8",
            #     "```"
            # ],
            "num_ctx": 2048
        },
        "keep_alive": 10000
    }
    chunk_count = 0
    start_time = time.perf_counter_ns()
    # print("Start")
    with session.post(url, json=payload, stream=True) as response:
        response.raise_for_status()
        first_byte_ns = time.perf_counter_ns()
        full_text = []
        objects = []
        for chunk in response.iter_content(chunk_size=None):
            if not chunk:
                continue
            chunk_count += 1
            for line in chunk.splitlines():         
                obj = json.loads(line)
                objects.append(obj)
                if "response" in obj:
                    if first_token_ns == 0:
                        first_token_ns = time.perf_counter_ns()
                    last_token_ns = time.perf_counter_ns()
                    full_text.append(obj["response"])
                if obj.get("done"):
                    break
    end_time = time.perf_counter_ns()
    # print("".join(full_text))
    input_token_counts = get_length_function("".join(prompt))
    output_token_counts = get_length_function("".join(full_text))
    # tps after first token
    if last_token_ns != first_byte_ns:
        tps = (output_token_counts-1) / ((last_token_ns - first_token_ns) / 1e9)
    else:
        tps = 0
    # print(token_counts, (last_token_ns - first_token_ns) / 1e9)
    return {
        "ttfb": (first_byte_ns - start_time) / 1e9,
        "ttft": (first_token_ns - start_time) / 1e9,
        "tps": tps,
        "input_token_counts": input_token_counts,
        "output_token_counts": output_token_counts,
        "latency": (end_time - start_time) / 1e9,
        "chunk_count": chunk_count
    }


In [ ]:
session = requests.Session()
session.trust_env = False

with open("test_code.py", 'r', encoding='utf-8') as file:
    text = file.read()
print(get_length_function(text))
def clear_cache(model_name: str):
    benchmark(URL, model_name, "#",session)
total_ids = TOKENIZER.encode(text, return_tensors=None, add_special_tokens=False)
def bench_mark_all(model_name: str):
    total_result = []
    for i in tqdm(range(0, len(total_ids), 8)):
        ids = total_ids[:i] 
        text = TOKENIZER.decode(ids)
        result = benchmark(URL, model_name, text, session)
        # print(result)
        total_result.append(result)
    return total_result
def bench_mark_all_no_cache(model_name: str):
    total_result = []
    for i in tqdm(range(0, len(total_ids), 8)):
        clear_cache(model_name)
        ids = total_ids[:i]
        text = TOKENIZER.decode(ids)
        result = benchmark(URL, model_name, text, session)
        # print(result)
        total_result.append(result)
    return total_result

In [ ]:
result_with_cache = {}
result_without_cache = {}
for model_name in model_names:
    clear_cache(model_name)
    res = bench_mark_all_no_cache(model_name)
    result_without_cache[model_name] = res
for model_name in model_names:
    clear_cache(model_name)
    res = bench_mark_all(model_name)
    result_with_cache[model_name] = res
    # print(res)

In [ ]:
import pickle
with open(save_file_name, 'wb') as file:
    data = {
        "cache": result_with_cache,
        "nocache": result_without_cache
    }
    pickle.dump(data, file)

In [ ]:
import pickle
with open(save_file_name, 'rb') as file:
    data = pickle.load(file)
    result_with_cache = data["cache"]
    result_without_cache = data["nocache"]

In [ ]:
def sample(values: list, step: int) -> list:
    indices = list(range(0, len(values), step))
    if indices[-1] != len(values) - 1:
        indices.append(len(values)-1)
    return [values[i] for i in indices]
for model_name in model_names:
    result_cache = result_with_cache[model_name][:DISPLAY_LIMIT//8]
    result_nocache = result_without_cache[model_name][:DISPLAY_LIMIT//8]
    latency_cache = [item['ttft'] * 1000 for item in result_cache]
    latency_nocache = [item['ttft'] * 1000 for item in result_nocache]
    
    import matplotlib.pyplot as plt

    # example data
    a = latency_cache[:]
    b = latency_nocache[:]
    # x-axis: indices
    x_a = [i * 8 for i in range(1, len(a)+1)]
    x_b = [i * 8 for i in range(1, len(b)+1)]
    x_ticks = sample(x_a, DISPLAY_SAMPLE_STEP)

    
    plt.figure(figsize=(10, 3))
    plt.plot(x_a, a, label="Cached (n-8 previous tokens)")
    plt.plot(x_b, b, label="Without cache")

    plt.xlabel("Context token counts")
    plt.ylabel("Time (ms)")
    min_y = min(min(a), min(b))
    max_y = max(max(a), max(b))
    
    # Annotate min and max
    plt.axhline(min_y, linestyle="--", alpha=0.5)
    plt.axhline(300, linestyle="--", alpha=0.5, color="green")
    plt.axhline(max_y, linestyle="--", alpha=0.5, color="red")


    ax = plt.gca()

    # Add space on the right for text
    plt.subplots_adjust(right=0.85)

    # Text outside plot (x in axes coords, y in data coords)
    ax.text(
        1.01, 0, f"Min: {min_y:.2f} ms",
        transform=ax.get_yaxis_transform(),
        va="bottom", ha="left", fontsize=9
    )

    ax.text(
        1.01, 300+max_y/10, f"300 ms",
        transform=ax.get_yaxis_transform(),
        va="top", ha="left", fontsize=9
    )

    ax.text(
        1.01, max_y, f"Max: {max_y:.2f} ms",
        transform=ax.get_yaxis_transform(),
        va="top", ha="left", fontsize=9
    )
    # Legend outside on the right
    ax.legend(
        loc="center left",
        bbox_to_anchor=(1.1, 0.5),
        frameon=False
    )
    
    
    plt.xticks(x_ticks)
    plt.ylim([0, max_y * 1.1])
    plt.xlim(8, len(a) * 8)
    plt.title("Time to first token")
    plt.tight_layout()

    plt.show()

In [ ]:
for model_name in model_names:
    result_cache = result_with_cache[model_name][:DISPLAY_LIMIT//8]
    result_nocache = result_without_cache[model_name][:DISPLAY_LIMIT//8]
    latency_cache = [item.get('latency', item.get('total_time')) * 1000 for item in result_cache]
    latency_nocache = [item.get('latency', item.get('total_time')) * 1000 for item in result_nocache]
    
    import matplotlib.pyplot as plt

    # example data
    a = latency_cache[:]
    b = latency_nocache[:]
    # x-axis: indices
    x_a = [i * 8 for i in range(1, len(a)+1)]
    x_b = [i * 8 for i in range(1, len(b)+1)]
    x_ticks = sample(x_a, DISPLAY_SAMPLE_STEP)

    
    plt.figure(figsize=(10, 3))
    plt.plot(x_a, a, label="Cached (n-8 previous tokens)")
    plt.plot(x_b, b, label="Without cache")

    plt.xlabel("Context token counts")
    plt.ylabel("Time (ms)")
    min_y = min(min(a), min(b))
    max_y = max(max(a), max(b))
    
    # Annotate min and max
    plt.axhline(min_y, linestyle="--", alpha=0.5)
    plt.axhline(300, linestyle="--", alpha=0.5, color="green")
    plt.axhline(max_y, linestyle="--", alpha=0.5, color="red")


    ax = plt.gca()

    # Add space on the right for text
    plt.subplots_adjust(right=0.85)

    # Text outside plot (x in axes coords, y in data coords)
    ax.text(
        1.01, 0, f"Min: {min_y:.2f} ms",
        transform=ax.get_yaxis_transform(),
        va="bottom", ha="left", fontsize=9
    )

    ax.text(
        1.01, 300+max_y/10, f"300 ms",
        transform=ax.get_yaxis_transform(),
        va="top", ha="left", fontsize=9
    )

    ax.text(
        1.01, max_y, f"Max: {max_y:.2f} ms",
        transform=ax.get_yaxis_transform(),
        va="top", ha="left", fontsize=9
    )
    # Legend outside on the right
    ax.legend(
        loc="center left",
        bbox_to_anchor=(1.1, 0.5),
        frameon=False
    )
    
    
    plt.xticks(x_ticks)
    plt.ylim([0, max_y * 1.1])
    plt.xlim(8, len(a) * 8)
    plt.title("Latency")
    plt.tight_layout()

    plt.show()
    

In [ ]:
for model_name in model_names:
    result_cache = result_with_cache[model_name][:DISPLAY_LIMIT//8]
    result_nocache = result_without_cache[model_name][:DISPLAY_LIMIT//8]
    import matplotlib.pyplot as plt
    tps_cache = [item['tps'] for item in result_cache]
    tps_nocache = [item['tps'] for item in result_nocache]
    tps_cache[0] = 0
    tps_nocache[0] = 0
    # example data
    a = tps_cache
    b = tps_nocache
    # x-axis: indices
    x_a = [i * 8 for i in range(1, len(a)+1)]
    x_b = [i * 8 for i in range(1, len(b)+1)]
    x_ticks = sample(x_a, DISPLAY_SAMPLE_STEP)

    
    plt.figure(figsize=(10, 3))

    plt.plot(x_a, tps_cache, label="Cached (n-8 previous tokens)")
    plt.plot(x_b, tps_nocache, label="Without cache")
    plt.xticks(x_ticks)
    plt.xlim(16, len(a) * 8)
    plt.legend()

    plt.xlabel("Context token counts")
    plt.ylabel("Count")
    plt.title("Tokens per second")
    plt.tight_layout()

    plt.show()

In [ ]:
import json
print(json.dumps(result_cache,ensure_ascii=False))